# Advanced Causal Estimators Demo

This notebook demonstrates advanced causal inference methods:
- **Regression Discontinuity Design (RDD)**
- **Synthetic Control Method**
- **Mediation Analysis**
- **Conditional Average Treatment Effects (CATE)**

In [ ]:
:dep datascienceutils-core = { path = "../datascienceutils-core", features = ["causal-analysis"] }
:dep ndarray = "0.15"
:dep rand = "0.8"

In [ ]:
use datascienceutils_core::analyze::*;
use ndarray::{array, Array1, Array2};
use rand::Rng;

## 1. Regression Discontinuity Design (RDD)

**Scenario:** Scholarship program with test score cutoff at 70.
- Students scoring ≥70 receive scholarship
- Outcome: College graduation rate

In [ ]:
// Generate RDD data
let mut rng = rand::thread_rng();
let n = 200;
let cutoff = 70.0;

let mut test_scores = Vec::new();
let mut graduation_rates = Vec::new();

for _ in 0..n {
    let score = rng.gen_range(50.0..90.0);
    
    // Baseline graduation rate increases with score
    let baseline = 0.3 + (score - 50.0) * 0.01;
    
    // Scholarship effect: +20% graduation rate
    let treatment_effect = if score >= cutoff { 0.20 } else { 0.0 };
    
    let grad_rate = baseline + treatment_effect + rng.gen_range(-0.05..0.05);
    
    test_scores.push(score);
    graduation_rates.push(grad_rate);
}

let running_var = Array1::from_vec(test_scores);
let outcome = Array1::from_vec(graduation_rates);

println!("=== Regression Discontinuity Design ===");
println!("Cutoff: {}", cutoff);
println!("Sample size: {}", n);

In [ ]:
// Estimate treatment effect at cutoff
let rdd_estimate = regression_discontinuity(&running_var, &outcome, cutoff, None).unwrap();

println!("\nRDD Results:");
println!("Estimated scholarship effect: {:.1}%", rdd_estimate * 100.0);
println!("True effect: 20.0%");
println!("Estimation error: {:.1}%", (rdd_estimate * 100.0 - 20.0).abs());

if (rdd_estimate * 100.0 - 20.0).abs() < 5.0 {
    println!("✓ RDD successfully recovered the true effect!");
}

## 2. Synthetic Control Method

**Scenario:** Policy intervention in California (2010)
- Compare California to synthetic control (weighted average of other states)
- Outcome: Economic growth rate

In [ ]:
// California (treated unit)
let ca_pre = array![2.1, 2.3, 2.2, 2.4];  // 2006-2009
let ca_post = array![3.5, 3.7, 3.6, 3.8]; // 2010-2013 (after policy)

// Control states (Texas, Florida, New York)
let control_pre = array![
    [2.0, 2.1, 2.0, 2.2],  // Texas
    [2.2, 2.3, 2.1, 2.3],  // Florida
    [2.1, 2.2, 2.2, 2.4],  // New York
];

let control_post = array![
    [2.1, 2.2, 2.1, 2.3],  // Texas (stable)
    [2.3, 2.4, 2.2, 2.4],  // Florida (stable)
    [2.2, 2.3, 2.3, 2.5],  // New York (stable)
];

println!("=== Synthetic Control Method ===");
println!("Treated unit: California");
println!("Control units: Texas, Florida, New York");
println!("Intervention year: 2010");

In [ ]:
let sc_estimate = synthetic_control(&ca_pre, &ca_post, &control_pre, &control_post).unwrap();

println!("\nSynthetic Control Results:");
println!("Estimated policy effect: {:.2}% growth", sc_estimate);
println!("\nInterpretation:");
if sc_estimate > 0.5 {
    println!("✓ Policy had a significant positive effect on California's economy");
} else {
    println!("⚠ Policy effect is small or uncertain");
}

## 3. Mediation Analysis

**Scenario:** Job training program
- **Treatment:** Job training
- **Mediator:** Skills improvement
- **Outcome:** Income increase

Question: How much of the income effect is mediated through skills?

In [ ]:
// Simulate mediation data
let training = array![0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0];

// Skills: affected by training
let skills = array![5.0, 6.0, 5.5, 8.0, 8.5, 9.0, 5.2, 8.2, 5.8, 8.8];

// Income: affected by both training and skills
let income = array![30.0, 32.0, 31.0, 45.0, 47.0, 50.0, 31.5, 46.0, 32.5, 48.0];

println!("=== Mediation Analysis ===");
println!("Treatment: Job training");
println!("Mediator: Skills improvement");
println!("Outcome: Income (thousands)");

In [ ]:
let (total_effect, direct_effect, indirect_effect) = 
    mediation_analysis(&training, &skills, &income).unwrap();

println!("\nMediation Results:");
println!("Total effect: ${:.2}k", total_effect);
println!("Direct effect: ${:.2}k", direct_effect);
println!("Indirect effect (via skills): ${:.2}k", indirect_effect);

let mediation_pct = (indirect_effect / total_effect) * 100.0;
println!("\n{:.1}% of the effect is mediated through skills improvement", mediation_pct);

if mediation_pct > 50.0 {
    println!("✓ Skills improvement is the primary mechanism!");
}

## 4. Conditional Average Treatment Effects (CATE)

**Scenario:** Drug effectiveness varies by patient age
- Analyze heterogeneous treatment effects across age groups

In [ ]:
// Generate heterogeneous treatment effect data
let confounders = array![
    [25.0, 120.0],  // Young, low BP
    [30.0, 125.0],
    [35.0, 130.0],
    [40.0, 135.0],  // Middle-aged
    [45.0, 140.0],
    [50.0, 145.0],
    [55.0, 150.0],  // Older
    [60.0, 155.0],
];

let treatment = array![0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0];

// Outcome: drug more effective for older patients
let outcome = array![
    70.0, 75.0,  // Young: small effect
    72.0, 80.0,  // Middle: moderate effect
    74.0, 88.0,  // Older: large effect
    76.0, 95.0,
];

println!("=== Conditional Average Treatment Effects ===");
println!("Analyzing treatment effect heterogeneity by age");

In [ ]:
let cate_results = conditional_ate(&confounders, &treatment, &outcome, 0).unwrap();

println!("\nCATE Results (by age quartile):");
for (i, (threshold, effect)) in cate_results.iter().enumerate() {
    println!("Quartile {}: Effect = {:.1} points", i + 1, effect);
}

println!("\n✓ Treatment effect increases with age!");
println!("Recommendation: Prioritize older patients for this treatment.");

## Summary

**Advanced Causal Methods Demonstrated:**

1. **RDD** - Credible estimates at discontinuities (scholarship cutoff)
2. **Synthetic Control** - Policy evaluation with aggregate data
3. **Mediation** - Decompose effects into direct/indirect pathways
4. **CATE** - Identify heterogeneous treatment effects

**When to use each:**
- **RDD**: Sharp cutoffs in treatment assignment
- **Synthetic Control**: Single treated unit, multiple controls
- **Mediation**: Understanding mechanisms of action
- **CATE**: Personalized treatment recommendations